# Kiểm tra snapshot đã xuất bản 28/09/2026
Đọc snapshot cố định, kiểm tra checksum, khử trùng và không chồng nhóm train/test. Không thay đổi raw hoặc MinIO.

In [ ]:
from pathlib import Path
import hashlib, json
from collections import Counter
root = Path.cwd()
if root.name == "notebooks": root = root.parent
snapshot = root / "data/processed/runs/8e58a94851f5f62c"
manifest = json.loads((snapshot / "manifest.json").read_text(encoding="utf-8"))
for name, expected in manifest["artifacts"].items():
    payload = (snapshot / name).read_bytes()
    assert len(payload) == expected["bytes"]
    assert hashlib.sha256(payload).hexdigest() == expected["sha256"]
rows = [json.loads(line) for line in (snapshot / "reviews.jsonl").read_text(encoding="utf-8").splitlines()]
quality = json.loads((snapshot / "quality_report.json").read_text(encoding="utf-8"))
assert len(rows) == len({r["review_id"] for r in rows})
assert quality["input_rows"] == len(rows) + quality.get("duplicate_ids_removed", 0) + quality.get("rejected_rows", 0)
groups = {}
for row in rows:
    groups.setdefault(row["split_group_id"], set()).add(row["split"])
assert all(len(splits) == 1 for splits in groups.values())
{"reviews": len(rows), "by_source": dict(Counter(r["source"] for r in rows)),
 "training_eligible": sum(r["training_eligible"] for r in rows), "checks": "passed"}
